# TP 3 — Validation croisée et GridSearchCV

**Objectifs**

- Comparer deux modèles par validation croisée plutôt que par un seul split.
- Utiliser `GridSearchCV` pour tuner un SVM.
- Lire les `cv_results_` et tracer une heatmap des scores.
- Évaluer le modèle final sur le test held-out.

**Durée indicative : 50 minutes.**

In [1]:
import matplotlib.pyplot as plt
import numpy as np
from sklearn.datasets import load_breast_cancer
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import GridSearchCV, cross_val_score, train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.svm import SVC

data = load_breast_cancer()
X, y = data.data, data.target  # binaire : 0 = malin, 1 = bénin
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, stratify=y, random_state=0)
print("train :", X_train.shape, "  test :", X_test.shape)

train : (455, 30)   test : (114, 30)


## Exercice 1 — Comparer deux modèles par CV

Avec `cross_val_score(cv=5, scoring="accuracy")`, comparez sur le train :

- un Pipeline `StandardScaler + SVC(kernel="rbf")` (par défaut),
- un `RandomForestClassifier(n_estimators=200, random_state=0)`.

Affichez moyenne ± écart-type pour chacun et concluez sur le meilleur baseline.

<details>
<summary>💡 Coup de pouce — comparer deux modèles par cross-validation</summary>

**🎯 But :** estimer la performance d'un modèle de façon **robuste** en moyennant sur plusieurs splits train/val.

**Utilisation de base**

```python
from sklearn.model_selection import cross_val_score
scores = cross_val_score(model, X_train, y_train, cv=5, scoring='accuracy')
```

`scores` est un tableau de **5 valeurs**, une par fold. Pour résumer :

```python
print(f"{scores.mean():.3f} ± {scores.std():.3f}")
```

**Pourquoi 5 folds ?**

- **2-3 folds** : peu coûteux mais haute variance des estimations.
- **5-10 folds** : compromis classique (5 par défaut dans sklearn récent).
- **Leave-one-out** : 1 fold par échantillon, ultra coûteux et peu informatif.

**Pas besoin de splitter manuellement avant**

Erreur fréquente : faire `train_test_split` puis appeler `cross_val_score(X_train, y_train)`. C'est OK mais réducteur — `cross_val_score` fait **ses propres splits internes** sur ce qu'on lui donne. Donner tout `X` est légitime si on n'a pas d'autre besoin de tenir un test set séparé.

**Stratification automatique**

Pour une classification, `cv=5` utilise par défaut `StratifiedKFold` (préserve les proportions de classes). Pour de la régression, c'est `KFold` simple.

**Comparer deux modèles**

```python
scores_a = cross_val_score(model_a, X_train, y_train, cv=5)
scores_b = cross_val_score(model_b, X_train, y_train, cv=5)
print(f"A : {scores_a.mean():.3f} ± {scores_a.std():.3f}")
print(f"B : {scores_b.mean():.3f} ± {scores_b.std():.3f}")
```

Si A.mean > B.mean **et** la différence > l'écart-type → A est probablement meilleur. Sinon, indistinguable au niveau de bruit observé.

</details>

In [2]:
# TODO


## Exercice 2 — GridSearchCV sur le SVM

1. Construisez un Pipeline `[StandardScaler, SVC(kernel="rbf")]`.
2. Définissez `param_grid = {"clf__C": [0.1, 1, 10, 100], "clf__gamma": ["scale", 0.01, 0.001]}`.
3. Lancez `GridSearchCV(pipe, param_grid, cv=5, scoring="accuracy", n_jobs=-1)`.
4. Affichez `best_params_`, `best_score_`, et l'accuracy sur le test held-out.

<details>
<summary>💡 Coup de pouce — GridSearchCV sur un Pipeline</summary>

**🎯 But :** explorer une grille d'hyperparamètres et choisir automatiquement la meilleure combinaison.

**Définir la grille**

```python
from sklearn.model_selection import GridSearchCV
param_grid = {
    'clf__C':     [0.1, 1, 10, 100],
    'clf__gamma': [0.001, 0.01, 0.1, 1],
}
```

⚠️ **Double underscore `clf__C`** : c'est la syntaxe pour cibler le paramètre `C` de l'étape nommée `'clf'` dans la Pipeline. Sans ça, sklearn ne sait pas à quelle étape ça s'applique.

Si le Pipeline était `Pipeline([('pre', PCA()), ('clf', SVC())])`, on pourrait écrire `'pre__n_components': [5, 10]` pour la PCA et `'clf__C': [...]` pour le SVM dans la même grille.

**Lancer la recherche**

```python
gs = GridSearchCV(pipe, param_grid, cv=5, scoring='accuracy', n_jobs=-1)
gs.fit(X_train, y_train)
```

- `n_jobs=-1` parallélise sur tous les cœurs disponibles.
- Total d'entraînements = `len(grid) * cv` = 4 × 4 × 5 = **80 modèles** entraînés ici.

**Récupérer les résultats**

```python
print(gs.best_params_)        # dict des meilleurs hyperparamètres
print(gs.best_score_)          # score CV moyen du meilleur modèle
gs.best_estimator_             # le Pipeline déjà refit sur tout X_train
gs.score(X_test, y_test)       # score final sur le test
```

⚠️ **Important** : par défaut `refit=True` → `gs.best_estimator_` est **refit sur 100 % de X_train** avec les meilleurs hyperparamètres, prêt à être utilisé directement.

</details>

In [3]:
# TODO


## Exercice 3 — Heatmap des scores CV

Récupérez `gs.cv_results_["mean_test_score"]`, reshape-le en `(len(C), len(gamma))`, et affichez-le sous forme de **heatmap** (`plt.imshow` + annotations).

Repérez visuellement la zone optimale.

<details>
<summary>💡 Coup de pouce — heatmap des scores CV</summary>

**🎯 But :** visualiser tous les scores de la grille pour comprendre où sont les régions « bonnes » d'hyperparamètres.

**Reformer les scores en grille 2D**

`gs.cv_results_['mean_test_score']` est un tableau **1D** dans l'ordre de la grille. Si la grille a 4 C × 4 gammas = 16 combinaisons, il faut reshape en (4, 4) :

```python
Cs = [0.1, 1, 10, 100]
gammas = [0.001, 0.01, 0.1, 1]
scores = gs.cv_results_['mean_test_score'].reshape(len(Cs), len(gammas))
```

⚠️ **Ordre du reshape** : il dépend de l'ordre de définition dans `param_grid`. Si `param_grid = {'clf__C': Cs, 'clf__gamma': gammas}`, alors `C` varie sur le 1er axe (lignes), `gamma` sur le 2e (colonnes).

**Afficher la heatmap**

```python
fig, ax = plt.subplots(figsize=(6, 5))
im = ax.imshow(scores, cmap='viridis', aspect='auto')
ax.set_xticks(range(len(gammas))); ax.set_xticklabels(gammas)
ax.set_yticks(range(len(Cs)));     ax.set_yticklabels(Cs)
ax.set_xlabel('gamma'); ax.set_ylabel('C')
plt.colorbar(im, ax=ax)
```

**Annoter chaque case avec sa valeur**

```python
for i in range(len(Cs)):
    for j in range(len(gammas)):
        ax.text(j, i, f"{scores[i,j]:.3f}", ha='center', va='center',
                color='white' if scores[i,j] < scores.mean() else 'black')
```

Le `color='white' if ... else 'black'` rend le texte lisible sur fond clair ET sombre.

**Ce que vous devriez observer**

Sur Digits avec SVM RBF, vous verrez typiquement une **bande diagonale** de bonnes performances. Le SVM RBF a une corrélation forte entre C et gamma : il faut les régler ensemble, pas isolément.

</details>

In [4]:
# TODO


## Exercice 4 — Évaluation finale

1. Refit le meilleur modèle sur tout le train (c'est ce que fait `GridSearchCV` par défaut avec `refit=True`).
2. Affichez le `classification_report` et la matrice de confusion sur le test.
3. Commentez : la précision et le rappel sont-ils équilibrés sur les deux classes ? Dans un contexte médical, lequel privilégier ?

<details>
<summary>💡 Coup de pouce — évaluation finale sur le test set</summary>

**🎯 But :** mesurer la performance « propre » du modèle sélectionné par GridSearch sur des données qu'il n'a JAMAIS vues.

**Le bon réflexe**

Toutes les décisions (choix d'hyperparamètres, comparaisons, sélection de modèle) ont été faites avec **CV sur le train**. Le **test set** est intouchable, gardé pour cette évaluation finale unique.

```python
print(f"CV score (train) : {gs.best_score_:.3f}")
print(f"Test score       : {gs.score(X_test, y_test):.3f}")
```

Si test ≈ CV → généralisation OK. Si test << CV → on a sur-optimisé sur la CV (rare avec une grille raisonnable, plus fréquent avec une grille énorme).

**Rapport de classification détaillé**

```python
from sklearn.metrics import classification_report
y_pred = gs.predict(X_test)
print(classification_report(y_test, y_pred, target_names=class_names))
```

Affiche **precision**, **recall**, **f1** pour chaque classe + moyenne. Plus informatif que l'accuracy seule.

- **Precision** : « sur ce que j'ai prédit positif, quelle fraction l'était vraiment ? »
- **Recall (rappel)** : « sur tous les vrais positifs, quelle fraction ai-je détectée ? »
- **F1** : moyenne harmonique des deux.

**Choisir entre precision et recall**

Dépend du coût des erreurs :
- **Spam** : précision prioritaire (rater du spam < classer un email légitime en spam).
- **Médical / fraude** : rappel prioritaire (rater un cas malade << fausse alerte qu'on peut vérifier).
- **Dataset déséquilibré** : toujours regarder f1 ET la matrice de confusion, jamais l'accuracy seule.

</details>

In [5]:
# TODO
